In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

# --- 辅助函数：计算临床评价指标 ---
def get_clinical_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    return sensitivity, specificity

# --- (1) 数据读取与清洗 ---
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", 
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]
df = pd.read_csv(url, names=columns, na_values='?')
df.dropna(inplace=True)
df["target"] = (df["target"] > 0).astype(int)

# --- (2) 标准化 ---
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(df.drop("target", axis=1)), columns=df.columns[:-1])
y = df["target"]

print(f"数据预处理完成。样本总数: {len(df)}")

数据预处理完成。样本总数: 297


In [13]:
# 1. 划分独立测试集 (30%)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 2. 模拟半监督划分 (20% Labeled / 80% Unlabeled)
X_labeled, X_unlabeled, y_labeled, _ = train_test_split(
    X_train_full, y_train_full, train_size=0.2, random_state=42, stratify=y_train_full
)

# 复制一份数据用于迭代，避免修改原始数据集
X_pool_labeled = X_labeled.copy().reset_index(drop=True)
y_pool_labeled = y_labeled.copy().reset_index(drop=True)
X_pool_unlabeled = X_unlabeled.copy().reset_index(drop=True)

# 初始化样本权重 (初始有标签数据权重为 1.0)
sample_weights = np.ones(len(X_pool_labeled))

print(f"初始有标签样本数: {len(X_pool_labeled)}")
print(f"初始无标签样本池: {len(X_pool_unlabeled)}")

初始有标签样本数: 41
初始无标签样本池: 166


In [14]:
# --- 自训练参数设置 ---
MAX_ITER = 10         # 最大迭代轮数
THRESHOLD = 0.85      # 置信度阈值 (严格筛选，宁缺毋滥)
model = LogisticRegression(max_iter=1000, random_state=42)

print(f"\n=== 开始自训练迭代 (Max Iter={MAX_ITER}, Threshold={THRESHOLD}) ===")

for i in range(MAX_ITER):
    # 1. 训练：基于当前扩充后的数据集训练模型
    # 关键点：每次迭代都会基于更新后的权重重新拟合
    model.fit(X_pool_labeled, y_pool_labeled, sample_weight=sample_weights)
    
    # 若无标签池已空，则提前结束
    if len(X_pool_unlabeled) == 0:
        break
        
    # 2. 预测：对剩余未标注数据进行推断
    probs = model.predict_proba(X_pool_unlabeled)
    confidence = probs.max(axis=1)           # 获取置信度
    predicted_class = probs.argmax(axis=1)   # 获取伪标签
    
    # 3. 筛选：仅保留高置信度样本
    high_conf_idx = np.where(confidence > THRESHOLD)[0]
    
    # 终止条件：若本轮无样本满足阈值，说明模型已收敛
    if len(high_conf_idx) == 0:
        print(f"Round {i+1}: 无新样本满足阈值，模型收敛。")
        break
        
    # 4. 迁移：将样本从"未标注池"移动到"有标签集"
    # 提取新样本
    X_new = X_pool_unlabeled.iloc[high_conf_idx]
    y_new = pd.Series(predicted_class[high_conf_idx], name="target")
    w_new = confidence[high_conf_idx] # 使用置信度作为权重
    
    # 更新有标签集合
    X_pool_labeled = pd.concat([X_pool_labeled, X_new], axis=0).reset_index(drop=True)
    y_pool_labeled = pd.concat([y_pool_labeled, y_new], axis=0).reset_index(drop=True)
    sample_weights = np.concatenate([sample_weights, w_new])
    
    # 更新未标注集合 (移除已选样本)
    remain_mask = ~X_pool_unlabeled.index.isin(high_conf_idx)
    X_pool_unlabeled = X_pool_unlabeled.iloc[remain_mask].reset_index(drop=True)
    
    print(f"Round {i+1}: 吸纳 {len(high_conf_idx)} 例伪标签 (剩余未标注: {len(X_pool_unlabeled)})")

print("=== 自训练迭代结束 ===")


=== 开始自训练迭代 (Max Iter=10, Threshold=0.85) ===
Round 1: 吸纳 80 例伪标签 (剩余未标注: 86)
Round 2: 吸纳 23 例伪标签 (剩余未标注: 63)
Round 3: 吸纳 11 例伪标签 (剩余未标注: 52)
Round 4: 吸纳 5 例伪标签 (剩余未标注: 47)
Round 5: 吸纳 4 例伪标签 (剩余未标注: 43)
Round 6: 吸纳 3 例伪标签 (剩余未标注: 40)
Round 7: 吸纳 2 例伪标签 (剩余未标注: 38)
Round 8: 吸纳 1 例伪标签 (剩余未标注: 37)
Round 9: 吸纳 1 例伪标签 (剩余未标注: 36)
Round 10: 吸纳 3 例伪标签 (剩余未标注: 33)
=== 自训练迭代结束 ===


In [15]:
# 1. 预测
y_pred_final = model.predict(X_test)
y_prob_final = model.predict_proba(X_test)[:, 1]

# 2. 计算指标
acc = accuracy_score(y_test, y_pred_final)
auc = roc_auc_score(y_test, y_prob_final)
sens, spec = get_clinical_metrics(y_test, y_pred_final)

print("\n=== 自训练模型最终表现 ===")
print(f"Accuracy:    {acc:.4f}")
print(f"AUC:         {auc:.4f}")
print(f"Sensitivity: {sens:.4f} (灵敏度)")
print(f"Specificity: {spec:.4f} (特异度)")
print(f"最终训练样本数: {len(X_pool_labeled)} (原始: {len(X_labeled)})")


=== 自训练模型最终表现 ===
Accuracy:    0.7778
AUC:         0.8542
Sensitivity: 0.7143 (灵敏度)
Specificity: 0.8333 (特异度)
最终训练样本数: 174 (原始: 41)
